# PySpark

## Customers

In [0]:
df  = spark.read.format('csv').\
        option("inferSchema",'true').\
        option("header",'true').\
        load('/Volumes/databricksmayank/bronze/bronze_volume/customers/dim_customers.csv')

display(df)

In [0]:
df2 = spark.read.csv('/Volumes/databricksmayank/bronze/bronze_volume/customers/dim_customers.csv', header=True, inferSchema=True)
display(df2)

In [0]:
df3 = spark.read.format('csv').options(inferSchema='true', header='true').load('/Volumes/databricksmayank/bronze/bronze_volume/customers/dim_customers.csv')
display(df3)

## Additional Options

### Option 4: Read with Explicit Schema (Best Practice)
Defining schema explicitly improves performance and data quality

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema explicitly
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

df4 = spark.read.format('csv')\
    .option("header", "true")\
    .schema(schema)\
    .load('/Volumes/databricksmayank/bronze/bronze_volume/customers/dim_customers.csv')

display(df4)

### Option 5: Auto Loader (Incremental Processing)
Optimal for production workloads with incremental data ingestion

In [0]:
# Auto Loader - incrementally processes new files as they arrive
df5 = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format', 'csv')\
    .option('cloudFiles.schemaLocation', '/Volumes/databricksmayank/bronze/bronze_volume/customers/_schema')\
    .option('header', 'true')\
    .load('/Volumes/databricksmayank/bronze/bronze_volume/customers/')

# Display streaming dataframe (shows schema and sample)
df5.printSchema()

### Option 6: SQL `read_files()` Function
Directly query files using SQL syntax

In [0]:
%sql
SELECT * 
FROM read_files(
  '/Volumes/databricksmayank/bronze/bronze_volume/customers/dim_customers.csv',
  format => 'csv',
  header => true
)
LIMIT 10

In [0]:
# Display streaming dataframe (shows schema and sample)
df.printSchema()

In [0]:
df6 = df.withColumnRenamed('customer_name', 'name')
display(df)
df7 = df.withColumnRenamed('email', 'email') 
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = df.withColumn("name", upper(col("name")))

In [0]:
display(df)

In [0]:
df = df.withColumn("domain",split(col("email"), "@")[1])
display(df)
df8 = df.withColumn("domain",regexp_replace(col("domain"), "\\.com$", ""))
display(df8)

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("total_customers")).sort("total_customers", ascending=False))

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("total_customers")).sort(col("total_customers").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumn("processDates",current_timestamp())
display(df)

In [0]:
__dw_df_0_in = spark.sql("""WITH q AS (select * from DatabricksViewf52810b) SELECT `domain`,`total_customers` FROM q GROUP BY `domain`,`total_customers`""")
__dw_df_0 = __dw_df_0_in.filter("(total_customers > 1)")

display(__dw_df_0)

#### UPSERT

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricksmayank.silver.customer_enr"):
    dlt_obj = DeltaTable.forName(spark, "databricksmayank.silver.customer_enr")
    dlt_obj.alias("trg").merge(df.alias("src"), "trg.customer_id = src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df.write.format("delta")\
        .mode("append")\
            .saveAsTable("databricksmayank.silver.customer_enr")

In [0]:
%sql
select * from databricksmayank.silver.customer_enr

### PRODUCTS

In [0]:
df_prod = spark.read.format("csv")\
        .option("inferSchema", "true")\
        .option("header", "true")\
        .load("/Volumes/databricksmayank/bronze/bronze_volume/products/")
display(df_prod)

In [0]:
df_prod = df_prod.withColumn("processDates",current_timestamp())
display(df_prod)

In [0]:
display(df_prod.groupBy("category").agg(avg("price").alias("avg_price")))

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
DROP TABLE IF EXISTS databricksmayank.silver.product_enr;

In [0]:
if spark.catalog.tableExists("databricksmayank.silver.product_enr"):
    dlt_obj = DeltaTable.forName(spark, "databricksmayank.silver.product_enr")
    dlt_obj.alias("trg").merge(df_prod.alias("src"), "trg.product_id = src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df_prod.write.format("delta")\
        .mode("append")\
            .saveAsTable("databricksmayank.silver.product_enr")

In [0]:
%sql
select * from databricksmayank.silver.product_enr

### STORES

In [0]:
df_store = spark.read.format("csv")\
        .option("inferSchema", "true")\
        .option("header", "true")\
        .load("/Volumes/databricksmayank/bronze/bronze_volume/stores/")
display(df_store)

In [0]:
from pyspark.sql.functions import regexp_replace, col
df_store = df_store.withColumn("store_name",regexp_replace(col("store_name"),"_",""))
df_store.write.mode("overwrite")
display(df_store)

In [0]:
from pyspark.sql.functions import current_timestamp
df_store = df_store.withColumn("processDates",current_timestamp())
display(df_store)

In [0]:
if spark.catalog.tableExists("databricksmayank.silver.stores_enr"):
    dlt_obj = DeltaTable.forName(spark, "databricksmayank.silver.stores_enr")
    dlt_obj.alias("trg").merge(df_store.alias("src"), "trg.product_id = src.storet_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df_store.write.format("delta")\
        .mode("append")\
            .saveAsTable("databricksmayank.silver.stores_enr")

In [0]:
%sql
select * from databricksmayank.silver.stores_enr

### Sales

In [0]:
df_sales = spark.read.format("csv")\
        .option("inferSchema", "true")\
        .option("header", "true")\
        .load("/Volumes/databricksmayank/bronze/bronze_volume/sales/")
display(df_sales)

In [0]:
from pyspark.sql.functions import current_timestamp, col, round
df_sales = df_sales.withColumn("pricePerSale",round(col("total_amount")/col("quantity"),2))
df_sales = df_sales.withColumn("processDates",current_timestamp())
display(df_sales)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricksmayank.silver.sales_enr"):
    dlt_obj = DeltaTable.forName(spark, "databricksmayank.silver.sales_enr")
    dlt_obj.alias("trg").merge(df_sales.alias("src"), "trg.sales_id = src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df_sales.write.format("delta")\
        .mode("append")\
            .saveAsTable("databricksmayank.silver.sales_enr")

In [0]:
%sql
select * from databricksmayank.silver.sales_enr

## SparkSQL

In [0]:
spark.sql("select * from databricksmayank.silver.customer_enr")

In [0]:
display(spark.sql("select * from databricksmayank.silver.customer_enr"))

In [0]:
df9 = spark.sql("select upper(category) category from databricksmayank.silver.product_enr")
display(df9)

In [0]:
df_prod.createOrReplaceTempView("temp_products")

In [0]:
df_prod_10 = spark.sql("""
          select *, 
                case 
                When category = 'Toys' then 'YES' else 'NO' end as flag
                from temp_products
          """)
display(df_prod_10)

### PySpark UDF


In [0]:
def greet(p_input):
    return "Hello " + p_input


In [0]:
udf_greet = udf(greet)


In [0]:
from pyspark.sql.functions import current_timestamp, col, round
df_prod_2 = df_prod_10.withColumn("greet",udf_greet(col("flag")))
display(df_prod_2)